In [ ]:
import json
from pathlib import Path

ROOT_DIR = Path("../").resolve()
print(f"ROOT_DIR: {ROOT_DIR}")

DATA_DIR = ROOT_DIR / "data"

RETRIEVAL_DIR = DATA_DIR / "retrieval"

QA_DATA_FILE = DATA_DIR / "qa" / "qa_data.json"

TASK_ID = "LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION"

DATA_COLUMNS = ["task_id", "input", "output"]

ROOT_DIR: C:\52300070_TMTHUYEN\LEARN_MORE\Apps\datn-legal-ai\backend


# Xây dựng các câu hỏi và bóc tách các trường thông tin từ câu hỏi
- Ví dụ: Điều, Khoản, Điểm, Chương, Phần, Tài liệu.

- Dữ liệu đề fine tune các câu hỏi truy xuất thông tin từ vị trí trích dẫn văn bản pháp luật.

In [22]:
import pandas as pd
import json
import random
from pathlib import Path 
from typing import Any
import re

In [23]:
LEGAL_QRHE_MULTI_SOURCE_FILE = RETRIEVAL_DIR / "LegalQRHE_multi_source.json"
LEGAL_QRHE_HIENPHAP_DANSU_FILE = RETRIEVAL_DIR / "LegalQRHE_hienphap_dansu.json"
LEGAL_QRHE_ALL_FILE = RETRIEVAL_DIR / "LegalQRHE_all.json"

In [ ]:
'''
Các text tra cưu luật: Bộ luật dân sự và Hiến pháp
Các nghị định, nghị quyết trong dữ liệu: 
metadata_source: "DOCX nguồn: van_ban/21_2021_ND-CP_468069.docx; effective_from: Điều 60 khoản 1"
21_2021_ND-CP_468069 
NGHỊ ĐỊNH
QUY ĐỊNH THI HÀNH BỘ LUẬT DÂN SỰ VỀ BẢO ĐẢM THỰC HIỆN NGHĨA VỤ
metadata_source: "DOCX nguồn: van_ban/19_2019_ND-CP_386324.docx; effective_from: Điều 26 khoản 1"
19_2019_ND-CP_386324
NGHỊ ĐỊNH
VỀ HỌ, HỤI, BIÊU, PHƯỜNG

'''

from src.common.custom_exception import CustomException
from src.retrieval.legal_query_extractor import LegalQueryExtractor
from src.utils.file_utils import FileUtils


qa_keys = ["question", "answer", "terms"]

def qa_to_input_retrievals(qa_data: list[dict]) -> list[dict]:
    """
    Convert QA data to input retrievals format.
    """
    def normalize_document(document: str) -> str:
        document = document.strip()

        # Loại bỏ các cụm dẫn chiếu ở đầu tên văn bản.
        document = re.sub(
            r"^\s*(?:(?:theo|tại|ở)\s+)?"
            r"(?:(?:điểm\s+[a-z])\s*,?\s*)?"
            r"(?:(?:khoản\s+\d+)\s*,?\s*)?"
            r"(?:(?:điều\s+\d+[a-z]?)\s*,?\s*)*",
            "",
            document,
            flags=re.IGNORECASE,
        )

        return document.strip(" ,:;.-")
    input_retrievals = []
    for qa in qa_data:
        document = qa.get("terms", "")
        query_text = document + ", " + qa.get("question", "")
        
        # normalize document
        document = normalize_document(document)
        
        if not document or not query_text:
            print(f"Skipping QA entry due to missing fields: {qa}")
            continue
        if query_text and document:
            input_retrievals.append({
                "query_text": query_text,
                "document": document
            })
            
    print(f"Converted {len(input_retrievals)} QA entries to input retrievals.")
    return input_retrievals
    

input_retrievals = [
    # --- 1. Nhóm cấu trúc chuẩn và các biến thể thông thường ---
    {
        "query_text": "Theo Điểm a, Khoản 2, Điều 3 của Bộ luật Dân sự năm 2015, bàn về cái gì?",
        "document": "Bộ luật Dân sự năm 2015"
    },
    {
        "query_text": "Theo Điều 5 của Hiến pháp năm 2013, quyền và nghĩa vụ cơ bản của công dân là gì?",
        "document": "Hiến pháp năm 2013"
    },
    {
        "query_text": "Theo Điểm b, Khoản 1, Điều 10 của Bộ luật Dân sự năm 2015, quy định về trách nhiệm dân sự như thế nào?",
        "document": "Bộ luật Dân sự năm 2015"
    },
    {
        "query_text": "Theo Điều 26 của Nghị định số 19/2019/NĐ-CP, quy định về việc gì?",
        "document": "Nghị định số 19/2019/NĐ-CP"
    },
    {
        "query_text": "Theo Điều 60 của Nghị định số 21/2021/NĐ-CP, quy định về việc gì?",
        "document": "Nghị định số 21/2021/NĐ-CP"
    },
    {
        "query_text": "Theo Tiểu mục 1, Mục 2, Chương III, Phần thứ nhất của Bộ luật Dân sự năm 2015, quy định điều gì?",
        "document": "Bộ luật Dân sự năm 2015"
    },
    {
        "query_text": "Theo Điểm c Khoản 3 Điều 45 của Luật Đất đai năm 2024 quy định như thế nào về bồi thường hỗ trợ?",
        "document": "Luật Đất đai năm 2024"
    },
    {
        "query_text": "Theo Khoản 4 Điều 8 của Luật Doanh nghiệp năm 2020 nghĩa vụ của doanh nghiệp là gì?",
        "document": "Luật Doanh nghiệp năm 2020"
    },
    {
        "query_text": "Theo Điểm a Khoản 1 Điều 2 của Luật Hôn nhân và gia đình năm 2014, nguyên tắc cơ bản là gì?",
        "document": "Luật Hôn nhân và gia đình năm 2014"
    },
    {
        "query_text": "Theo Mục 3 Chương II Phần thứ tư của Bộ luật Hình sự năm 2015 quy định về tội phạm kinh tế ra sao?",
        "document": "Bộ luật Hình sự năm 2015"
    },
    {
        "query_text": "Theo Khoản 2 Điều 155 của Bộ luật Lao động năm 2019 quyền lợi của lao động nữ được quy định thế nào?",
        "document": "Bộ luật Lao động năm 2019"
    },
    {
        "query_text": "Theo Điều 10 của Luật Các tổ chức tín dụng năm 2024 quy định hành vi bị nghiêm cấm nào?",
        "document": "Luật Các tổ chức tín dụng năm 2024"
    },
    {
        "query_text": "Theo Điểm d, Khoản 2, Điều 12 của Luật Đầu tư năm 2020 hình thức ưu đãi đầu tư gồm những gì?",
        "document": "Luật Đầu tư năm 2020"
    },
    {
        "query_text": "Theo Khoản 1 Điều 32 của Luật Xây dựng năm 2014 quy định về phân loại công trình như thế nào?",
        "document": "Luật Xây dựng năm 2014"
    },
    {
        "query_text": "Theo Tiểu mục 2 Mục 1 Chương I của Luật Giáo dục năm 2019 hệ thống giáo dục quốc dân gồm những gì?",
        "document": "Luật Giáo dục năm 2019"
    },
    {
        "query_text": "Theo Khoản 3 Điều 74 của Luật Thương mại năm 2005 hình thức chế tài gồm những gì?",
        "document": "Luật Thương mại năm 2005"
    },
    {
        "query_text": "Theo Điểm e Khoản 1 Điều 9 của Luật Thuế thu nhập cá nhân năm 2007 quy định các khoản giảm trừ nào?",
        "document": "Luật Thuế thu nhập cá nhân năm 2007"
    },
    {
        "query_text": "Theo Điều 14 của Luật Nhà ở năm 2023 chính sách phát triển nhà ở xã hội được nêu ra sao?",
        "document": "Luật Nhà ở năm 2023"
    },
    {
        "query_text": "Theo Khoản 5 Điều 21 của Luật Bảo hiểm xã hội năm 2024 quyền của người lao động là gì?",
        "document": "Luật Bảo hiểm xã hội năm 2024"
    },
    {
        "query_text": "Theo Điểm a Khoản 2 Điều 100 của Luật Đất đai năm 2013 điều kiện cấp sổ đỏ là gì?",
        "document": "Luật Đất đai năm 2013"
    },

    # --- 2. Nhóm số hiệu ban hành phức tạp (Nghị định, Thông tư ngành dọc, Quốc hội) ---
    {
        "query_text": "Theo Điều 12 của Nghị định số 100/2019/NĐ-CP xử phạt vi phạm giao thông thế nào?",
        "document": "Nghị định số 100/2019/NĐ-CP"
    },
    {
        "query_text": "Theo Khoản 2 Điều 5 của Thông tư số 12/2021/TT-BGTVT quy định về đào tạo lái xe ra sao?",
        "document": "Thông tư số 12/2021/TT-BGTVT"
    },
    {
        "query_text": "Theo Điều 3 của Thông tư số 200/2014/TT-BTC nguyên tắc kế toán được định nghĩa thế nào?",
        "document": "Thông tư số 200/2014/TT-BTC"
    },
    {
        "query_text": "Theo Điểm b Khoản 4 Điều 16 của Nghị định số 43/2014/NĐ-CP hướng dẫn thi hành luật đất đai thế nào?",
        "document": "Nghị định số 43/2014/NĐ-CP"
    },
    {
        "query_text": "Theo Điều 8 của Thông tư số 66/2026/TT-BGDĐT tiêu chuẩn trang thiết bị dạy học là gì?",
        "document": "Thông tư số 66/2026/TT-BGDĐT"
    },
    {
        "query_text": "Theo Khoản 1 Điều 2 của Nghị định số 15/2020/NĐ-CP quy định xử phạt bưu chính viễn thông thế nào?",
        "document": "Nghị định số 15/2020/NĐ-CP"
    },
    {
        "query_text": "Theo Điều 22 của Thông tư số 32/2023/TT-BYT quy định phạm vi hành nghề khám chữa bệnh ra sao?",
        "document": "Thông tư số 32/2023/TT-BYT"
    },
    {
        "query_text": "Theo Điểm a Khoản 3 Điều 9 của Nghị định số 119/2018/NĐ-CP quy định về hóa đơn điện tử thế nào?",
        "document": "Nghị định số 119/2018/NĐ-CP"
    },
    {
        "query_text": "Theo Điều 4 của Thông tư số 01/2021/TT-BKHĐT hướng dẫn đăng ký doanh nghiệp thế nào?",
        "document": "Thông tư số 01/2021/TT-BKHĐT"
    },
    {
        "query_text": "Theo Khoản 2 Điều 11 của Nghị định số 20/2019/NĐ-CP lệ phí trước bạ được tính ra sao?",
        "document": "Nghị định số 20/2019/NĐ-CP"
    },
    {
        "query_text": "Theo Điều 7 của Thông tư số 111/2013/TT-BTC căn cứ tính thuế thu nhập cá nhân gồm những gì?",
        "document": "Thông tư số 111/2013/TT-BTC"
    },
    {
        "query_text": "Theo Điểm c Khoản 1 Điều 14 của Nghị định số 46/2016/NĐ-CP xử phạt đường sắt thế nào?",
        "document": "Nghị định số 46/2016/NĐ-CP"
    },
    {
        "query_text": "Theo Điều 5 của Thông tư số 03/2017/TT-BTTTT tiêu chuẩn an toàn thông tin mạng cấp độ 3 là gì?",
        "document": "Thông tư số 03/2017/TT-BTTTT"
    },
    {
        "query_text": "Theo Khoản 3 Điều 25 của Nghị định số 08/2022/NĐ-CP hướng dẫn Luật Bảo vệ môi trường ra sao?",
        "document": "Nghị định số 08/2022/NĐ-CP"
    },
    {
        "query_text": "Theo Điều 18 của Thông tư số 22/2021/TT-BGDĐT cách đánh giá kết quả rèn luyện học sinh phổ thông là gì?",
        "document": "Thông tư số 22/2021/TT-BGDĐT"
    },
    {
        "query_text": "Theo Điểm b Khoản 2 Điều 6 của Nghị định số 96/2023/NĐ-CP điều kiện cơ sở khám bệnh là gì?",
        "document": "Nghị định số 96/2023/NĐ-CP"
    },
    {
        "query_text": "Theo Điều 15 của Thông tư số 09/2024/TT-BXD định mức dự toán xây dựng công trình được sửa đổi thế nào?",
        "document": "Thông tư số 09/2024/TT-BXD"
    },
    {
        "query_text": "Theo Khoản 1 Điều 30 của Nghị định số 136/2020/NĐ-CP quy định về phòng cháy chữa cháy thế nào?",
        "document": "Nghị định số 136/2020/NĐ-CP"
    },
    {
        "query_text": "Theo Điều 10 của Thông tư số 39/2016/TT-NHNN quy định về hoạt động cho vay thế nào?",
        "document": "Thông tư số 39/2016/TT-NHNN"
    },
    {
        "query_text": "Theo Điểm a Khoản 4 Điều 2 của Nghị định số 123/2020/NĐ-CP thời điểm lập hóa đơn là khi nào?",
        "document": "Nghị định số 123/2020/NĐ-CP"
    },

    # --- 3. Nhóm đảo thứ tự cấu trúc văn bản (Văn bản lên đầu hoặc câu hỏi tự do) ---
    {
        "query_text": "Trách nhiệm hình sự được quy định thế nào trong Bộ luật Hình sự năm 2015 tại Điều 12?",
        "document": "Bộ luật Hình sự năm 2015"
    },
    {
        "query_text": "Luật Đất đai năm 2024 tại Khoản 2 Điều 45 quy định gì về đất nông nghiệp?",
        "document": "Luật Đất đai năm 2024"
    },
    {
        "query_text": "Hãy tra cứu Thông tư số 12/2021/TT-BGTVT xem điều kiện kinh doanh vận tải xe khách là gì.",
        "document": "Thông tư số 12/2021/TT-BGTVT"
    },
    {
        "query_text": "Hiến pháp năm 2013 tại Điều 54 quy định đất đai thuộc sở hữu toàn dân như thế nào?",
        "document": "Hiến pháp năm 2013"
    },
    {
        "query_text": "Quyền đơn phương chấm dứt hợp đồng lao động tại Điều 35 Bộ luật Lao động năm 2019 được hiểu ra sao?",
        "document": "Bộ luật Lao động năm 2019"
    },
    {
        "query_text": "Nghị định số 100/2019/NĐ-CP tại Điều 16 có mức phạt nồng độ cồn là bao nhiêu?",
        "document": "Nghị định số 100/2019/NĐ-CP"
    },
    {
        "query_text": "Luật Doanh nghiệp năm 2020 ở Điều 17 cấm những tổ chức nào thành lập công ty?",
        "document": "Luật Doanh nghiệp năm 2020"
    },
    {
        "query_text": "Tại Điều 201 Bộ luật Hình sự năm 2015 cấu thành tội cho vay lãi nặng trong giao dịch dân sự là gì?",
        "document": "Bộ luật Hình sự năm 2015"
    },
    {
        "query_text": "Đọc kỹ Luật Thuế giá trị gia tăng năm 2008 tại Điều 5 để biết đối tượng không chịu thuế.",
        "document": "Luật Thuế giá trị gia tăng năm 2008"
    },
    {
        "query_text": "Xem hướng dẫn tại Nghị định số 43/2014/NĐ-CP Điều 20 về việc cấp giấy chứng nhận quyền sử dụng đất.",
        "document": "Nghị định số 43/2014/NĐ-CP"
    },
    {
        "query_text": "Thông tư số 200/2014/TT-BTC quy định tài khoản 111 tiền mặt ở Điều 12 thế nào?",
        "document": "Thông tư số 200/2014/TT-BTC"
    },
    {
        "query_text": "Bộ luật Dân sự năm 2015 bảo vệ quyền nhân thân tại Điều 25 bằng cách nào?",
        "document": "Bộ luật Dân sự năm 2015"
    },
    {
        "query_text": "Nghị định số 15/2020/NĐ-CP quy định hành vi tung tin giả trên mạng bị phạt ra sao tại Điều 101?",
        "document": "Nghị định số 15/2020/NĐ-CP"
    },
    {
        "query_text": "Tìm hiểu Luật Trẻ em năm 2016 tại Điều 6 về các hành vi bị nghiêm cấm đối với trẻ em.",
        "document": "Luật Trẻ em năm 2016"
    },
    {
        "query_text": "Theo quy định của Luật Giao thông đường bộ năm 2008 tại Điều 9 người tham gia giao thông phải đi thế nào?",
        "document": "Luật Giao thông đường bộ năm 2008"
    },
    {
        "query_text": "Căn cứ vào Nghị định số 21/2021/NĐ-CP ở Điều 60 để giải quyết tranh chấp hợp đồng bảo đảm.",
        "document": "Nghị định số 21/2021/NĐ-CP"
    },
    {
        "query_text": "Luật Phòng chống tác hại của rượu bia năm 2019 quy định địa điểm không uống rượu tại Điều 10 gồm những đâu?",
        "document": "Luật Phòng chống tác hại của rượu bia năm 2019"
    },
    {
        "query_text": "Bộ luật Tố tụng Dân sự năm 2015 quy định thời hiệu khởi kiện tại Điều 184 như thế nào?",
        "document": "Bộ luật Tố tụng Dân sự năm 2015"
    },
    {
        "query_text": "Nghiên cứu Luật Sở hữu trí tuệ năm 2005 tại Điều 14 về các loại hình tác phẩm được bảo hộ.",
        "document": "Luật Sở hữu trí tuệ năm 2005"
    },
    {
        "query_text": "Nghị định số 19/2019/NĐ-CP tại Điều 26 quy định về hình thức họ, hụ, biêu, phường ra sao?",
        "document": "Nghị định số 19/2019/NĐ-CP"
    },

    # --- 4. Nhóm biến thể tên văn bản không chứa năm hoặc dùng ký hiệu đặc biệt ---
    {
        "query_text": "Theo Điều 1 của Luật Ban hành văn bản quy phạm pháp luật quy định hệ thống văn bản gồm những gì?",
        "document": "Luật Ban hành văn bản quy phạm pháp luật"
    },
    {
        "query_text": "Theo Khoản 2 Điều 4 của Luật Nghĩa vụ quân sự độ tuổi gọi nhập ngũ được quy định thế nào?",
        "document": "Luật Nghĩa vụ quân sự"
    },
    {
        "query_text": "Theo Điều 7 của Luật Công nghệ thông tin các hành vi bị nghiêm cấm bao gồm những gì?",
        "document": "Luật Công nghệ thông tin"
    },
    {
        "query_text": "Theo Khoản 1 Điều 9 của Luật Cư trú điều kiện đăng ký thường trú mới nhất là gì?",
        "document": "Luật Cư trú"
    },
    {
        "query_text": "Theo Điều 13 của Luật Luật sư tiêu chuẩn luật sư tại Việt Nam cần những gì?",
        "document": "Luật Luật sư"
    },
    {
        "query_text": "Theo Khoản 3 Điều 5 của Luật Hải quan địa bàn hoạt động hải quan bao gồm những khu vực nào?",
        "document": "Luật Hải quan"
    },
    {
        "query_text": "Theo Điều 12 của Luật Thanh niên đối thoại với thanh niên được tổ chức định kỳ ra sao?",
        "document": "Luật Thanh niên"
    },
    {
        "query_text": "Theo Khoản 2 Điều 10 của Luật Du lịch điều kiện công nhận khu du lịch quốc gia là gì?",
        "document": "Luật Du lịch"
    },
    {
        "query_text": "Theo Điều 14 của Luật Công chứng giá trị pháp lý của văn bản công chứng được định nghĩa thế nào?",
        "document": "Luật Công chứng"
    },
    {
        "query_text": "Theo Khoản 1 Điều 3 của Luật Thủy sản quy định về phát triển bền vững ngành thủy sản thế nào?",
        "document": "Luật Thủy sản"
    },
    {
        "query_text": "Theo Điều 8 của Luật Điện lực các hành vi phá hoại an toàn điện lực bị xử lý ra sao?",
        "document": "Luật Điện lực"
    },
    {
        "query_text": "Theo Khoản 2 Điều 6 của Luật Kế toán các hành vi bị nghiêm cấm trong kế toán bao gồm những gì?",
        "document": "Luật Kế toán"
    },
    {
        "query_text": "Theo Điều 11 của Luật Thống kê quy định hệ thống thông tin thống kê quốc gia gồm những gì?",
        "document": "Luật Thống kê"
    },
    {
        "query_text": "Theo Khoản 4 Điều 7 của Luật Trọng tài thương mại thẩm quyền của Trọng tài được xác định thế nào?",
        "document": "Luật Trọng tài thương mại"
    },
    {
        "query_text": "Theo Điều 9 của Luật Hòa giải ở cơ sở nguyên tắc tổ chức hòa giải gồm những gì?",
        "document": "Luật Hòa giải ở cơ sở"
    },
    {
        "query_text": "Theo Khoản 2 Điều 5 của Luật Việc làm chính sách của Nhà nước về việc làm được nêu thế nào?",
        "document": "Luật Việc làm"
    },
    {
        "query_text": "Theo Điều 10 của Luật An toàn thông tin mạng phân loại cấp độ an toàn hệ thống ra sao?",
        "document": "Luật An toàn thông tin mạng"
    },
    {
        "query_text": "Theo Khoản 1 Điều 13 của Luật Tín ngưỡng tôn giáo quyền tự do tín ngưỡng quy định thế nào?",
        "document": "Luật Tín ngưỡng tôn giáo"
    },
    {
        "query_text": "Theo Điều 6 của Luật Quốc tịch Việt Nam giấy tờ chứng minh quốc tịch Việt Nam gồm loại nào?",
        "document": "Luật Quốc tịch Việt Nam"
    },
    {
        "query_text": "Theo Khoản 3 Điều 15 của Luật Đo lường quy định về kiểm định phương tiện đo lường ra sao?",
        "document": "Luật Đo lường"
    },

    # --- 5. Nhóm kiểm thử nâng cao (Dấu câu bất thường, thiếu dấu phẩy, viết tắt loại văn bản) ---
    {
        "query_text": "Theo Điều 3 Luật Giá năm 2023, nguyên tắc quản lý giá của Nhà nước được thực hiện ra sao?",
        "document": "Luật Giá năm 2023"
    },
    {
        "query_text": "Tra cứu giúp tôi Điều 45 Luật Kinh doanh bất động sản năm 2023 xem điều kiện chuyển nhượng dự án là gì.",
        "document": "Luật Kinh doanh bất động sản năm 2023"
    },
    {
        "query_text": "Theo Khoản 1 Điều 9 Bộ luật Hàng hải Việt Nam năm 2015 quy định về tàu biển như thế nào?",
        "document": "Bộ luật Hàng hải Việt Nam năm 2015"
    },
    {
        "query_text": "Hỏi về mức phạt xử lý vi phạm theo Nghị định số 45/2022/NĐ-CP ở Khoản 2 Điều 12.",
        "document": "Nghị định số 45/2022/NĐ-CP"
    },
    {
        "query_text": "Luật Hợp tác xã năm 2023 quy định điều kiện trở thành thành viên tại Điều 30 như thế nào?",
        "document": "Luật Hợp tác xã năm 2023"
    },
    {
        "query_text": "Quy định về kiểm soát quyền lực tại Điều 5 Luật Tổ chức Quốc hội năm 2014 được hiểu ra sao?",
        "document": "Luật Tổ chức Quốc hội năm 2014"
    },
    {
        "query_text": "Căn cứ Khoản 3 Điều 88 Hiến pháp năm 2013, Chủ tịch nước có thẩm quyền gì trong quân đội?",
        "document": "Hiến pháp năm 2013"
    },
    {
        "query_text": "Theo Điều 2 Luật Cạnh tranh năm 2018 đối tượng áp dụng của luật này bao gồm những ai?",
        "document": "Luật Cạnh tranh năm 2018"
    },
    {
        "query_text": "Điều 14 Luật An ninh mạng năm 2018 quy định về phòng chống gián điệp mạng như thế nào?",
        "document": "Luật An ninh mạng năm 2018"
    },
    {
        "query_text": "Khoản 2 Điều 7 Luật Kiến trúc năm 2019 bản sắc văn hóa dân tộc trong kiến trúc quy định ra sao?",
        "document": "Luật Kiến trúc năm 2019"
    },
    {
        "query_text": "Theo Điều 5 Luật Tố cáo năm 2018 nguyên tắc giải quyết tố cáo bao gồm những gì?",
        "document": "Luật Tố cáo năm 2018"
    },
    {
        "query_text": "Luật Chăn nuôi năm 2018 quy định hành vi nhân bản vô tính động vật tại Điều 12 bị cấm thế nào?",
        "document": "Luật Chăn nuôi năm 2018"
    },
    {
        "query_text": "Tìm hiểu Khoản 1 Điều 15 Luật Trồng trọt năm 2018 về bảo vệ quyền lợi người nông dân.",
        "document": "Luật Trồng trọt năm 2018"
    },
    {
        "query_text": "Điều 8 Luật Thủy lợi năm 2017 quy định các loại công trình thủy lợi được phân cấp ra sao?",
        "document": "Luật Thủy lợi năm 2017"
    },
    {
        "query_text": "Theo Luật Tiếp cận thông tin năm 2016 ở Điều 3, công dân được tiếp cận những thông tin nào?",
        "document": "Luật Tiếp cận thông tin năm 2016"
    },
    {
        "query_text": "Luật Dược năm 2016 quy định chính sách của Nhà nước về phát triển ngành dược tại Điều 7 thế nào?",
        "document": "Luật Dược năm 2016"
    },
    {
        "query_text": "Tra cứu Điều 9 Luật Thú y năm 2015 về danh mục bệnh động vật phải công bố dịch.",
        "document": "Luật Thú y năm 2015"
    },
    {
        "query_text": "Khoản 2 Điều 10 Luật An toàn vệ sinh lao động năm 2015 quy định quyền lợi người lao động ra sao?",
        "document": "Luật An toàn vệ sinh lao động năm 2015"
    },
    {
        "query_text": "Theo Điều 4 Luật MTTQ Việt Nam năm 2015 quy định quyền tham gia xây dựng Đảng thế nào?",
        "document": "Luật Mặt trận Tổ quốc Việt Nam năm 2015"
    },
    {
        "query_text": "Hãy phân tích Điều 12 Luật Thực hành tiết kiệm chống lãng phí năm 2013 quy định về mua sắm công.",
        "document": "Luật Thực hành tiết kiệm, chống lãng phí năm 2013"
    }
]

qa_data = FileUtils.load_json_file(str(QA_DATA_FILE))
qa_keys =["question", "answer", "terms"]
# check key
for key in qa_data[0].keys():
    if key not in qa_keys:
        print(f"Warning: Unexpected key '{key}' found in QA data.")
        
input_retrievals.extend(qa_to_input_retrievals(qa_data))
print(f"Loaded {len(input_retrievals)} input retrievals for testing.")

def extract_retrieval_data():
    extractor = LegalQueryExtractor()
    
    final_data = []
    
    for query in input_retrievals:
        try:
            query_text = query.get("query_text", "")
            document = query.get("document", "")
            
            result = extractor.extract(query_text=query_text, document=document)
            
            # check keys
            if not all(k in result for k in ["query_text", "hierarchy"]):
                raise CustomException(f"Missing expected keys in result: {result}")
            
            query_text = result.get("query_text", "")
            hierarchy = result.get("hierarchy", {})
            
            # print(f"Query: {query_text}\nQuestion: {question_text}\nhierarchy: {hierarchy}\n")
            final_data.append({
                "task_id": TASK_ID,
                "input": query_text,
                "output": hierarchy
            })
        except CustomException as e:
            print(f"Query: {query}\nError: {e.to_dict()}\n")
        except Exception as e:
            print(f"Query: {query}\nUnexpected Error: {str(e)}\n")
    return final_data

data = extract_retrieval_data()
FileUtils.save_json_file(data=data, dir=RETRIEVAL_DIR, file_name=LEGAL_QRHE_MULTI_SOURCE_FILE.name)


Converted 2519 QA entries to input retrievals.
Loaded 2619 input retrievals for testing.
Query: {'query_text': 'Theo khoản 6 Điều 16 Nghị định số 100/2019/NĐ-CP, Hành vi sử dụng biển số xe ô tô giả khi tham gia giao thông sẽ bị xử lý như thế nào?', 'document': 'Điều 16 Nghị định số 100/2019/NĐ-CP'}
Error: {'message': 'Khoản không thể tồn tại nếu không có Điều.', 'status_code': 400, 'error_code': 'CUSTOM_ERROR'}

Query: {'query_text': 'Theo Khoản 12 Điểu 2 Nghị định 123/2021/NĐ-CP, Có phạt tiền khi tài xế taxi không mặc đồng phục không?', 'document': 'Điểu 2 Nghị định 123/2021/NĐ-CP'}
Error: {'message': 'Khoản không thể tồn tại nếu không có Điều.', 'status_code': 400, 'error_code': 'CUSTOM_ERROR'}

Query: {'query_text': 'Theo Khoản 12 Điểu 2 Nghị định 123/2021/NĐ-CP, Có phạt tiền khi tài xế taxi không mặc đồng phục không?', 'document': 'Điểu 2 Nghị định 123/2021/NĐ-CP'}
Error: {'message': 'Khoản không thể tồn tại nếu không có Điều.', 'status_code': 400, 'error_code': 'CUSTOM_ERR

WindowsPath('C:/52300070_TMTHUYEN/LEARN_MORE/Apps/datn-legal-ai/backend/data/retrieval/LegalQRHE_multi_source.json')

In [25]:
HIERARCHY_FIELDS = [
    "part",
    "chapter",
    "section",
    "subsection",
    "article",
    "clause",
    "point",
]


class LegalTreeQueryGenerator:
    def __init__(self, seed: int = 42):
        """
        Duyệt cây pháp luật thô json
        Sinh ra các câu hỏi truy vấn luật pháp lý dựa trên các cấp của văn bản.
        Ouput: query_text, hierarchy
        """
        self.rng = random.Random(seed)

        # Mỗi template xác định rõ các cấp xuất hiện trong query.
        self.templates = [
            (
                ("part",),
                [
                    "Phần {part} của {document} quy định nội dung gì?",
                    "Nội dung Phần {part} của {document} là gì?",
                ],
            ),
            (
                ("chapter",),
                [
                    "Chương {chapter} của {document} quy định về vấn đề gì?",
                    "Nội dung của Chương {chapter} của {document} là gì?",
                ],
            ),
            (
                ("part", "chapter"),
                [
                    "Chương {chapter} thuộc Phần {part} của {document} quy định về vấn đề gì?",
                    "Trong Phần {part}, Chương {chapter} theo {document} có nội dung gì?",
                ],
            ),
            (
                ("section",),
                [
                    "Mục {section} của {document} quy định về vấn đề gì?",
                    "Nội dung của Mục {section} của {document} là gì?",
                ],
            ),
            (
                ("subsection",),
                [
                    "Tiểu mục {subsection} trong {document} quy định về vấn đề gì?",
                ],
            ),
            (
                ("article",),
                [
                    "Điều {article} của {document} quy định về vấn đề gì?",
                    "Nội dung Điều {article} theo {document} là gì?",
                ],
            ),
            (
                ("article", "clause"),
                [
                    "Khoản {clause} Điều {article} theo {document} quy định gì?",
                    "Nội dung Khoản {clause} Điều {article} theo {document} là gì?",
                ],
            ),
            (
                ("article", "point"),
                [
                    "Điểm {point} Điều {article} theo {document} quy định gì?",
                ],
            ),
            (
                ("article", "clause", "point"),
                [
                    "Điểm {point} Khoản {clause} Điều {article} theo {document} quy định gì?",
                    "Nội dung Điểm {point} Khoản {clause} Điều {article} theo {document} là gì?",
                ],
            ),
            (
                ("chapter", "article"),
                [
                    "Điều {article} thuộc Chương {chapter}  theo {document} quy định gì?",
                ],
            ),
            (
                ("part", "chapter", "article"),
                [
                    "Điều {article} thuộc Chương {chapter}, Phần {part} theo {document} quy định gì?",
                ],
            ),
            (
                ("part", "chapter", "article", "clause"),
                [
                    "Khoản {clause} Điều {article}, thuộc Chương {chapter} Phần {part}, theo {document} quy định gì?",
                ],
            ),
            (
                (
                    "part",
                    "chapter",
                    "article",
                    "clause",
                    "point",
                ),
                [
                    "Điểm {point} Khoản {clause} Điều {article}, thuộc Chương {chapter} Phần {part}, theo {document} quy định gì?",
                ],
            ),
        ]

    @staticmethod
    def get_document_name(data: dict[str, Any]) -> str:
        root = data.get("cau_truc", data)
        return (
            root.get("ten_van_ban")
            or root.get("ten")
            or root.get("so_van_ban")
            or "Văn bản pháp luật"
        )

    @staticmethod
    def normalize_number(value: Any) -> str | None:
        if value is None:
            return None

        value = str(value).strip()
        return value or None

    def walk(
        self,
        node: dict[str, Any],
        path: dict[str, str | None],
        document: str,
    ):
        """
        Duyệt cây pháp luật.
        path lưu vị trí thực tế của node trong cây.
        """
        child_fields = {
            "cac_phan": ("part",),
            "cac_chuong": ("chapter",),
            "cac_muc": ("section",),
            "cac_tieu_muc": ("subsection",),
            "cac_dieu": ("article",),
            "cac_khoan": ("clause",),
            "cac_diem": ("point",),
        }

        # Phát sinh mẫu tại các node có thể được trích dẫn.
        node_type = node.get("loai")
        if node_type in {
            "phan", "chuong", "muc", "tieu_muc",
            "dieu", "khoan", "diem",
        }:
            yield from self.emit_for_path(path, document)

        for child_key, (field,) in child_fields.items():
            children = node.get(child_key) or []

            for child in children:
                if not isinstance(child, dict):
                    continue

                child_path = path.copy()
                child_path[field] = self.normalize_number(
                    child.get("so")
                )

                yield from self.walk(
                    child,
                    child_path,
                    document,
                )

    def emit_for_path(
        self,
        path: dict[str, str | None],
        document: str,
    ):
        for fields, templates in self.templates:
            # Chỉ sinh nếu tất cả các cấp trong template
            # tồn tại thực tế trên đường dẫn cây.
            if not all(path.get(field) is not None for field in fields):
                continue

            hierarchy = {
                field: path.get(field) if field in fields else None
                for field in HIERARCHY_FIELDS
            }

            template = self.rng.choice(templates)
            query_text = template.format(**hierarchy, document=document)

            # Tách document thành trường riêng.
            yield {
                "query_text": query_text,
                "hierarchy": {
                    **hierarchy,
                    "document": document,
                },
            }

    def generate(self, data: dict[str, Any]) -> list[dict[str, Any]]:
        """ 
        Tạo các cập query_text và hierarchy tại mỗi node từ cây pháp luật 
        """
        root = data.get("cau_truc", data)
        document = self.get_document_name(data)

        initial_path = {
            field: None for field in HIERARCHY_FIELDS
        }

        samples = list(
            self.walk(root, initial_path, document)
        )

        # Loại bỏ các mẫu trùng hoàn toàn.
        unique = {}
        for sample in samples:
            key = (
                sample["query_text"],
                json.dumps(
                    sample["hierarchy"],
                    ensure_ascii=False,
                    sort_keys=True,
                ),
            )
            unique[key] = sample

        return list(unique.values())



def generate_legal_query(input_path: Path, output_path: Path):
    data = FileUtils.load_json_file(input_path)
    
    generator = LegalTreeQueryGenerator()
    samples = []
    
    for entry in data:
        entry_samples = generator.generate(entry)
        
        # task_id, input, output
        for sample in entry_samples:
            if not sample["query_text"] or not sample["hierarchy"]:
                print(f"Warning: Missing query_text or hierarchy in sample: {sample}")
                continue
            
            samples.append({
                "task_id": TASK_ID,
                "input": sample["query_text"],
                "output": sample["hierarchy"]
            })
        

    FileUtils.save_json_file(data=samples, dir=output_path.parent, file_name=output_path.name)

    print(f"Đã sinh {len(samples)} mẫu: {output_path}")
    
    return samples
    
    

legal_query = generate_legal_query(
    input_path=DATA_DIR / "du_lieu_phap_luat.json",
    output_path=LEGAL_QRHE_HIENPHAP_DANSU_FILE,
)
 

Data saved to C:\52300070_TMTHUYEN\LEARN_MORE\Apps\datn-legal-ai\backend\data\retrieval\LegalQRHE_hienphap_dansu.json
Đã sinh 7932 mẫu: C:\52300070_TMTHUYEN\LEARN_MORE\Apps\datn-legal-ai\backend\data\retrieval\LegalQRHE_hienphap_dansu.json


In [26]:

retrieval_data = []
for file_path in [
    LEGAL_QRHE_HIENPHAP_DANSU_FILE,
    LEGAL_QRHE_MULTI_SOURCE_FILE,
]:
    if not file_path.exists():
        print(f"File {file_path} không tồn tại. Vui lòng kiểm tra lại.")
        continue

    data = FileUtils.load_json_file(file_path)
    print(f"File {file_path} có {len(data)} mẫu.")
    
    retrieval_data.extend(data)     
 
FileUtils.save_json_and_csv(
    data=retrieval_data,
    dir=LEGAL_QRHE_ALL_FILE.parent, 
    file_name=LEGAL_QRHE_ALL_FILE.name
)
# Thống kế
df = pd.DataFrame(retrieval_data)

display(df.head(1000))
display(df.info())

File C:\52300070_TMTHUYEN\LEARN_MORE\Apps\datn-legal-ai\backend\data\retrieval\LegalQRHE_hienphap_dansu.json có 7932 mẫu.
File C:\52300070_TMTHUYEN\LEARN_MORE\Apps\datn-legal-ai\backend\data\retrieval\LegalQRHE_multi_source.json có 2616 mẫu.
Data saved to C:\52300070_TMTHUYEN\LEARN_MORE\Apps\datn-legal-ai\backend\data\retrieval\LegalQRHE_all.json
Data saved to C:\52300070_TMTHUYEN\LEARN_MORE\Apps\datn-legal-ai\backend\data\retrieval\LegalQRHE_all.json
Data saved to C:\52300070_TMTHUYEN\LEARN_MORE\Apps\datn-legal-ai\backend\data\retrieval\LegalQRHE_all.csv


,task_id,input,output
0,LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION,Chương I của Hiến pháp nước Cộng hòa xã hội ch...,"{'part': None, 'chapter': 'I', 'section': None..."
1,LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION,Nội dung Điều 1 theo Hiến pháp nước Cộng hòa x...,"{'part': None, 'chapter': None, 'section': Non..."
2,LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION,Điều 1 thuộc Chương I theo Hiến pháp nước Cộn...,"{'part': None, 'chapter': 'I', 'section': None..."
3,LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION,Điều 2 của Hiến pháp nước Cộng hòa xã hội chủ ...,"{'part': None, 'chapter': None, 'section': Non..."
4,LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION,Điều 2 thuộc Chương I theo Hiến pháp nước Cộn...,"{'part': None, 'chapter': 'I', 'section': None..."
...,...,...,...
995,LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION,Nội dung Khoản 3 Điều 36 theo Bộ luật Dân sự 2...,"{'part': None, 'chapter': None, 'section': Non..."
996,LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION,"Khoản 3 Điều 36, thuộc Chương III Phần thứ nhấ...","{'part': 'thứ nhất', 'chapter': 'III', 'sectio..."
997,LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION,Nội dung Điều 37 theo Bộ luật Dân sự 2015 là gì?,"{'part': None, 'chapter': None, 'section': Non..."
998,LEGAL_QUERY_REFERENCE_HIERARCHY_EXTRACTION,Điều 37 thuộc Chương III theo Bộ luật Dân sự ...,"{'part': None, 'chapter': 'III', 'section': No..."


<class 'pandas.DataFrame'>
RangeIndex: 10548 entries, 0 to 10547
Data columns (total 3 columns):
 #   Column   Non-Null Count  Dtype 
---  ------   --------------  ----- 
 0   task_id  10548 non-null  str   
 1   input    10548 non-null  str   
 2   output   10548 non-null  object
dtypes: object(1), str(2)
memory usage: 247.3+ KB


None